# L-1 Phase 2：日報深度標籤

對 `phase_l_final.csv`（1,802,590 筆、178,790 家公司）中每家公司：
- 依其擁有的 L 層，各取最近 3 篇日報作為上下文
- **1 次 LLM call / 公司**，同時萃取所有 L 層的深度結構標籤

| L 層 | 萃取欄位 |
|------|---------|
| L1 起因/阻礙 | pain_type, impact, urgency |
| L2 角色壓力 | role, pressure_source, kpi |
| L3 戰略目標 | goal_type, timeline, core_kpi |
| L4 產業議題 | issue_name, driver, stance |
| L5 問項精煉 | eval_items, competitors, decision_criteria |
| L6 動態屬性 | direction, trigger, current_temp |
| L7 實戰對策 | outcome_type, key_factor, next_step |

**前提**：`phase_l_final.csv` 已存在，SQL Server 可連線。

## 0. 安裝套件

In [1]:
!pip install google-genai pyodbc pandas tqdm

## 1. 匯入套件 & 全域設定

In [2]:
import os, re, json, time, configparser, threading
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd

# ── 路徑設定 ──────────────────────────────────────────
BASE_DIR      = Path(r"D:\yujui\痛點需求地圖")
PHASE_L_CSV   = BASE_DIR / "step3_output"    / "phase_l_final.csv"
OUTPUT_DIR    = BASE_DIR / "phase2_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_JSONL  = OUTPUT_DIR / "phase2_deep_labels.jsonl"
FLAT_CSV      = OUTPUT_DIR / "phase2_labels_flat.csv"

# ── SQL 設定 ──────────────────────────────────────────
SQL_INI   = r"D:\yujui\SqlServer18.txt"
SQL_TABLE = "DSC_CRM_SP.dbo.CRMGY"
TOP_LOGS_PER_LAYER = 3   # 每家公司每個 L 層取最近 N 篇

# ── LLM 設定 ──────────────────────────────────────────
MODEL      = "gemini-2.5-flash"  # ✅ 已驗證可用（Vertex AI）
TEMP       = 0.1
RPM_LIMIT  = 100                 # ✅ 安全範圍（避免 429）
N_WORKERS  = 15                  # ✅ 平衡效率與穩定性
RESUME     = True

print("設定完成")
print(f"  PHASE_L_CSV : {PHASE_L_CSV}")
print(f"  OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"  MODEL={MODEL}  RPM={RPM_LIMIT}  WORKERS={N_WORKERS}")

d:\miniforge3\envs\yujui_even\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


設定完成
  PHASE_L_CSV : D:\yujui\痛點需求地圖\step3_output\phase_l_final.csv
  OUTPUT_DIR  : D:\yujui\痛點需求地圖\phase2_output
  MODEL=gemini-2.5-flash  RPM=100  WORKERS=15


## 2. 載入設定 & 連線

In [3]:
import pyodbc
from google import genai
from google.genai import types as genai_types

GCP_INI = r"D:\yujui\GoogleCloud.ini"
cfg = configparser.ConfigParser()
cfg.read(GCP_INI, encoding="utf-8")

def _get_ini(cfg, key):
    for sec in list(cfg.sections()) + ['DEFAULT']:
        for k in (key, f"'{key}'"):
            try:
                v = cfg.get(sec, k)
                if v: return v.strip("'\" ")
            except: pass
    raise KeyError(f'{key} not found')

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:/Users/DSC/yujui/auto-upload-dataset/digiwin-ai-cf22a107ca03.json"
PROJECT_ID = _get_ini(cfg, 'project_id')  # digiwin-ai
client = genai.Client(vertexai=True, project=PROJECT_ID, location='us-central1')
print(f"Vertex AI 連線成功（project={PROJECT_ID}）")

# ── SQL 連線 ────────────────────────────────────────────────
sql_cfg = configparser.ConfigParser()
sql_cfg.read(SQL_INI, encoding="utf-8")
sec = sql_cfg["mssql"]
CONN_STR = (
    f"DRIVER={{SQL Server}};SERVER={sec['server']};"
    f"DATABASE=DSC_CRM_SP;UID={sec['uid']};PWD={sec['pwd']}"
)
conn = pyodbc.connect(CONN_STR, autocommit=True)
print(f"SQL 連線成功：{sec['server']}")


Vertex AI 連線成功（project=digiwin-ai）
SQL 連線成功：10.20.99.18


---
## 子任務 A：載入 phase_l_final.csv & 建立公司→L層索引

In [4]:
# ── A1：載入分類結果，建立 company_id → {l_layer: [ym]} 索引 ──
df = pd.read_csv(PHASE_L_CSV,
                 usecols=['company_id','ym','l_layer'],
                 dtype={'company_id':str,'ym':str,'l_layer':str},
                 low_memory=False)
df = df[df['l_layer'].isin(['L1','L2','L3','L4','L5','L6','L7'])]

# 每家公司每個 L 層取最近 TOP_LOGS_PER_LAYER 個 ym
company_index = defaultdict(lambda: defaultdict(list))
for _, row in df.sort_values('ym', ascending=False).iterrows():
    cid, ym, layer = row['company_id'], row['ym'], row['l_layer']
    if len(company_index[cid][layer]) < TOP_LOGS_PER_LAYER:
        company_index[cid][layer].append(ym)

ALL_COMPANIES = list(company_index.keys())
print(f'唯一公司數：{len(ALL_COMPANIES):,} 家')
print(f'預估 LLM calls：{len(ALL_COMPANIES):,} 次')
print(f'預估耗時（RPM={RPM_LIMIT}）：{len(ALL_COMPANIES)/RPM_LIMIT/60:.0f} 小時')

唯一公司數：170,945 家
預估 LLM calls：170,945 次
預估耗時（RPM=100）：28 小時


In [5]:
# ── A2：Resume（只跳過成功記錄，_error 重跑）──
done_ids = set()
if RESUME and RESULT_JSONL.exists():
    with open(RESULT_JSONL, encoding="utf-8") as f:
        for l in f:
            if not l.strip(): continue
            r = json.loads(l)
            if "_error" not in r.get("labels", {}) and r.get("labels"):
                done_ids.add(r["company_id"])
    print(f"已完成（成功）：{len(done_ids):,} 家")
    print(f"將重跑錯誤/空標籤：{len(ALL_COMPANIES)-len(done_ids):,} 家")
else:
    if not RESUME and RESULT_JSONL.exists():
        os.remove(RESULT_JSONL)
    print(f"從頭開始，共 {len(ALL_COMPANIES):,} 家")

TODO_COMPANIES = [c for c in ALL_COMPANIES if c not in done_ids]
print(f"本次待處理：{len(TODO_COMPANIES):,} 家")


已完成（成功）：161,246 家
將重跑錯誤/空標籤：9,699 家
本次待處理：9,699 家


---
## 子任務 B：SQL 查詢 log_content + LLM 深度萃取

In [6]:
# ── B1：SQL 查詢函數（批次查詢指定公司的日報）────────────
_sql_lock = threading.Lock()

def fetch_logs(company_id: str, layer_yms: dict) -> dict:
    """回傳 {l_layer: [log_content, ...]}"""
    result = defaultdict(list)
    for layer, yms in layer_yms.items():
        if not yms: continue
        
        # ✅ 修正：GY003 是日期（YYYYMMDD），ym 是 "YYYY-MM"
        # 轉換 "2026-04" -> "202604" 來比對 GY003 前綴
        ym_prefixes = [ym.replace('-', '') for ym in yms]  # "2026-04" -> "202604"
        
        # 使用 LIKE 比對前綴（避免 IN 條件，直接抓該年月的所有日報）
        like_conditions = ' OR '.join(f"GY003 LIKE '{prefix}%'" for prefix in ym_prefixes)
        
        sql = (
            f"SELECT TOP {TOP_LOGS_PER_LAYER} GY012 FROM {SQL_TABLE} "
            f"WHERE GY001='{company_id}' "
            f"AND ({like_conditions}) "
            f"AND LEN(GY012)>=10 ORDER BY GY003 DESC"
        )
        
        with _sql_lock:
            cur = conn.cursor()
            cur.execute(sql)
            rows = cur.fetchall()
        result[layer] = [r[0].replace('\n',' ')[:300] for r in rows if r[0]]
    return dict(result)

print('B1 SQL 函數定義完成（已修正 GY003 日期欄位）')

B1 SQL 函數定義完成（已修正 GY003 日期欄位）


In [7]:
# ── B2：Prompt 與 LLM 呼叫函數 ─────────────────────────
SYSTEM_PROMPT = (
    "你是業務日誌深度萃取專家。根據各 L 層業務日誌，萃取結構化深度標籤。\n\n"
    "各層萃取欄位：\n"
    "L1（起因/阻礙）：pain_type（阻礙類型）, impact（影響範疇）, urgency（高/中/低）\n"
    "L2（角色壓力）：role（角色名稱）, pressure_source（壓力來源）, kpi（指標）\n"
    "L3（戰略目標）：goal_type（目標類型）, timeline（時間範圍）, core_kpi（核心指標）\n"
    "L4（產業議題）：issue_name（議題名稱）, driver（外部驅動力）, stance（積極/觀望/防禦）\n"
    "L5（問項精煉）：eval_items（評估項目列表）, competitors（比較對象）, decision_criteria（決策條件）\n"
    "L6（動態屬性）：direction（升溫/降溫/持平）, trigger（觸發事件）, current_temp（熱/暖/冷）\n"
    "L7（實戰對策）：outcome_type（簽約/報價/拒絕/延後）, key_factor（關鍵因素）, next_step（後續跟進）\n\n"
    "只回傳該公司實際有的 L 層，格式：\n"
    "{\"L1\": {\"pain_type\": \"...\", ...}, \"L3\": {...}}\n"
    "若某欄位無資訊填 null。"
)

_JSON_PAT = re.compile(r'\{[\s\S]+\}', re.DOTALL)

def build_prompt(logs_by_layer: dict) -> str:
    parts = ["以下是該公司各 L 層的業務日誌：\n"]
    for layer in ['L1','L2','L3','L4','L5','L6','L7']:
        logs = logs_by_layer.get(layer, [])
        if not logs: continue
        parts.append(f'【{layer}】')
        for i, log in enumerate(logs, 1):
            parts.append(f'  {i}. {log[:200]}')
    return "\n".join(parts)

def call_llm(company_id: str, logs_by_layer: dict) -> dict:
    prompt = build_prompt(logs_by_layer)
    if not prompt.strip() or len(logs_by_layer) == 0:
        return {}
    for attempt in range(5):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=genai_types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=TEMP,
                    response_mime_type='application/json',
                ),
            )
            m = _JSON_PAT.search(resp.text.strip())
            return json.loads(m.group()) if m else {}
        except Exception as e:
            err = str(e)
            if '429' in err or 'RESOURCE_EXHAUSTED' in err:
                wait = min(30 * (2 ** attempt), 300)
                time.sleep(wait)
                continue
            return {'_error': err}
    return {'_error': 'max retries (429)'}

print('B2 LLM 函數定義完成（含 retry）')

B2 LLM 函數定義完成（含 retry）


In [8]:
# ── B3：RateLimiter + 多執行緒批次萃取 ─────────────────
class RateLimiter:
    def __init__(self, rpm):
        self._lock = threading.Lock()
        self._times = []
        self._interval = 60.0 / rpm
    def wait(self):
        with self._lock:
            now = time.time()
            self._times = [t for t in self._times if now - t < 60]
            if len(self._times) >= RPM_LIMIT:
                sleep_t = 60 - (now - self._times[0]) + 0.1
                time.sleep(max(sleep_t, 0))
            self._times.append(time.time())

rl = RateLimiter(RPM_LIMIT)
_write_lock = threading.Lock()

def process_company(company_id):
    layer_yms = dict(company_index[company_id])
    logs_by_layer = fetch_logs(company_id, layer_yms)
    rl.wait()
    labels = call_llm(company_id, logs_by_layer)
    rec = {
        'company_id':    company_id,
        'l_layers':      list(layer_yms.keys()),
        'labels':        labels,
        'processed_at':  datetime.now().isoformat(),
    }
    with _write_lock:
        with open(RESULT_JSONL, 'a', encoding='utf-8') as f:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    return company_id

written = errors = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(process_company, cid): cid for cid in TODO_COMPANIES}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='深度標籤'):
        try:
            fut.result()
            written += 1
        except Exception as e:
            errors += 1

print(f'\n完成：{written:,} 家  錯誤：{errors}')
print(f"結果 → {RESULT_JSONL}")

深度標籤: 100%|██████████| 9699/9699 [1:37:47<00:00,  1.65it/s]  


完成：9,698 家  錯誤：1
結果 → D:\yujui\痛點需求地圖\phase2_output\phase2_deep_labels.jsonl


---
## 子任務 E：統計 & 輸出

In [9]:
# ── E1：統計報告 ────────────────────────────────────────
records = [json.loads(l) for l in open(RESULT_JSONL, encoding='utf-8') if l.strip()]
total = len(records)
print(f'已處理：{total:,} 家')

# 各 L 層萃取覆蓋率
layer_cnt = defaultdict(int)
error_cnt = 0
for r in records:
    lb = r.get('labels', {})
    if '_error' in lb:
        error_cnt += 1
        continue
    for layer in ['L1','L2','L3','L4','L5','L6','L7']:
        if layer in lb and lb[layer]:
            layer_cnt[layer] += 1

print(f'\n各 L 層深度標籤覆蓋：')
for layer in ['L1','L2','L3','L4','L5','L6','L7']:
    n = layer_cnt[layer]
    pct = n/total*100 if total else 0
    print(f'  {layer}  {n:>8,} 家  ({pct:.1f}%)')
print(f'\n錯誤（_error）：{error_cnt}')

已處理：173,376 家

各 L 層深度標籤覆蓋：
  L1   127,126 家  (73.3%)
  L2    87,983 家  (50.7%)
  L3    57,046 家  (32.9%)
  L4    28,751 家  (16.6%)
  L5    75,264 家  (43.4%)
  L6    44,863 家  (25.9%)
  L7    39,334 家  (22.7%)

錯誤（_error）：1154


In [10]:
# ── E2：輸出 phase2_labels_flat.csv ─────────────────────
import pandas as pd

rows = []
for r in records:
    lb = r.get('labels', {})
    if '_error' in lb: continue
    base = {'company_id': r['company_id']}
    for layer in ['L1','L2','L3','L4','L5','L6','L7']:
        if layer not in lb or not lb[layer]: continue
        sub = lb[layer] if isinstance(lb[layer], dict) else {}
        for k, v in sub.items():
            base[f'{layer}_{k}'] = v
    rows.append(base)

flat_df = pd.DataFrame(rows)
flat_df.to_csv(FLAT_CSV, index=False, encoding='utf-8-sig')
print(f"✅ phase2_deep_labels.jsonl → {RESULT_JSONL}  ({total:,} 筆)")
print(f"✅ phase2_labels_flat.csv  → {FLAT_CSV}  ({len(flat_df):,} 行)")
print(f'   欄位數：{len(flat_df.columns)}')
flat_df.head(3)

✅ phase2_deep_labels.jsonl → D:\yujui\痛點需求地圖\phase2_output\phase2_deep_labels.jsonl  (173,376 筆)
✅ phase2_labels_flat.csv  → D:\yujui\痛點需求地圖\phase2_output\phase2_labels_flat.csv  (172,222 行)
   欄位數：92


,company_id,L5_eval_items,L5_competitors,L5_decision_criteria,L7_outcome_type,L7_key_factor,L7_next_step,L3_goal_type,L3_timeline,L3_core_kpi,...,L7_cost_explanation,L2_1,L2_2,L2_3,L5_1,L5_2,L5_價格與優惠,L5_內部簽核進度,L7_產品功能與價格,L7_客戶戰略需求 (IPO)
0,8200506423,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000135517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0000698635,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
